# Phase 2 - Feature Engineering and Mood Labels

notebook loads the clean Phase 1 data and adds the features needed for Moodwave analysis.

The main mood label uses Spotify **valence** and **energy**:

- high valence + high energy = Euphoric
- high valence + low energy = Peaceful
- low valence + high energy = Aggressive
- low valence + low energy = Melancholic

In [1]:
import os
import json
import pandas as pd
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working folder:", os.getcwd())

Working folder: C:\dev\Moodwave\moodwave-ml


## 1. Load the clean Phase 1 files

In [2]:
# Use Parquet when it exists, otherwise use the compressed CSV copy.
if os.path.exists('data/interim/historical_tracks_clean.parquet'):
    historical = pd.read_parquet('data/interim/historical_tracks_clean.parquet')
    genre = pd.read_parquet('data/interim/genre_tracks_clean.parquet')
    country = pd.read_parquet('data/interim/country_chart_clean.parquet')
else:
    historical = pd.read_csv('data/interim/historical_tracks_clean.csv.gz')
    genre = pd.read_csv('data/interim/genre_tracks_clean.csv.gz')
    country = pd.read_csv('data/interim/country_chart_clean.csv.gz')

country['snapshot_date'] = pd.to_datetime(country['snapshot_date'], errors='coerce')
country['album_release_date'] = pd.to_datetime(country['album_release_date'], errors='coerce')

print("Historical:", historical.shape)
print("Genre:", genre.shape)
print("Country:", country.shape)

Historical: (2400, 23)
Genre: (113550, 21)
Country: (2110316, 25)


In [3]:
historical.head()

,playlist_url,year,track_id,track_name,popularity,album_name,artist_id,artist_name,artist_genres,artist_popularity,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,6naxalmIoLFWR0siv8dnQQ,Oops!...I Did It Again,81,Oops!... I Did It Again,26dSoYclwsYLMAKD3tpOr4,Britney Spears,"['dance pop', 'pop']",81,...,-5.444,0,0.0437,0.3000,0.000018,0.3550,0.894,95.053,211160,4
1,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,2m1hi0nfMR9vdGC8UcrnwU,All The Small Things,83,Enema Of The State,6FBDaR13swtiWwGhX1WQsP,blink-182,"['alternative metal', 'modern rock', 'pop punk...",79,...,-4.918,1,0.0488,0.0103,0.000000,0.6120,0.684,148.726,167067,4
2,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,3y4LxiYMgDl4RethdzpmNe,Breathe,66,Breathe,25NQNriVT2YbSW80ILRWJa,Faith Hill,"['contemporary country', 'country', 'country d...",62,...,-9.007,1,0.0290,0.1730,0.000000,0.2510,0.278,136.859,250547,4
3,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,0v1XpBHnsbkCn7iJ9Ucr1l,It's My Life,81,Crush,58lV9VcRSjABbAbfWS6skp,Bon Jovi,"['glam metal', 'rock']",79,...,-4.063,0,0.0466,0.0263,0.000013,0.3470,0.544,119.992,224493,4
4,https://open.spotify.com/playlist/37i9dQZF1DWU...,2000,62bOmKYxYg7dhrC6gH9vFn,Bye Bye Bye,75,No Strings Attached,6Ff53KvcvAj5U7Z1vojB5o,*NSYNC,"['boy band', 'dance pop', 'pop']",70,...,-4.843,0,0.0479,0.0310,0.001200,0.0821,0.861,172.638,200400,4


In [4]:
genre.head()

,source_row_id,track_id,artist_name,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [5]:
country.head()

,track_id,track_name,artist_name,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,explicit,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,Alex Warren,1,1,0,GLOBAL,2025-06-11,95,False,...,2,-6.141,1,0.0600,0.704000,0.000007,0.0550,0.391,168.115,3
1,42UBPzRMh5yyz0EDPr6fr1,Manchild,Sabrina Carpenter,2,-1,48,GLOBAL,2025-06-11,89,True,...,7,-5.087,1,0.0572,0.122000,0.000000,0.3170,0.811,123.010,4
2,0FTmksd2dxiE5e3rWyJXs6,back to friends,sombr,3,0,1,GLOBAL,2025-06-11,98,False,...,1,-2.291,1,0.0301,0.000094,0.000088,0.0929,0.235,92.855,4
3,7so0lgd0zP2Sbgs2d7a1SZ,Die With A Smile,"Lady Gaga, Bruno Mars",4,0,-1,GLOBAL,2025-06-11,91,False,...,6,-7.727,0,0.0317,0.289000,0.000000,0.1260,0.498,157.964,3
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,0,GLOBAL,2025-06-11,100,False,...,2,-10.171,1,0.0358,0.200000,0.060800,0.1170,0.438,104.978,4


## 2. Create mood labels

In [6]:
# Conditions use 0.50 as the high/low boundary for both valence and energy.
def add_mood_columns(df):
    conditions = [
        (df['valence'] >= 0.50) & (df['energy'] >= 0.50),
        (df['valence'] >= 0.50) & (df['energy'] < 0.50),
        (df['valence'] < 0.50) & (df['energy'] >= 0.50),
        (df['valence'] < 0.50) & (df['energy'] < 0.50)
    ]

    moods = ['Euphoric', 'Peaceful', 'Aggressive', 'Melancholic']

    # adding 3 new colums related to mood classification
    df['mood_label'] = np.select(conditions, moods, default='Unknown')
    df['mood_valence_band'] = np.where(df['valence'] >= 0.50, 'high', 'low')
    df['mood_energy_band'] = np.where(df['energy'] >= 0.50, 'high', 'low')

    return df

historical = add_mood_columns(historical)
genre = add_mood_columns(genre)
country = add_mood_columns(country)

In [7]:
print("Historical mood counts:")
print(historical['mood_label'].value_counts())

print("\nGenre mood counts:")
print(genre['mood_label'].value_counts())

Historical mood counts:
mood_label
Euphoric       1259
Aggressive      815
Melancholic     241
Peaceful         85
Name: count, dtype: int64

Genre mood counts:
mood_label
Euphoric       43296
Aggressive     38666
Melancholic    22910
Peaceful        8678
Name: count, dtype: int64


## 3. Add simple useful features

In [8]:
# Duration is easier to understand in minutes.
historical['duration_min'] = historical['duration_ms'] / 60000
genre['duration_min'] = genre['duration_ms'] / 60000
country['duration_min'] = country['duration_ms'] / 60000

# Historical data is grouped into decades for later comparisons.
historical['release_decade'] = (historical['year'] // 10) * 10

# Rank 1 becomes score 50, rank 50 becomes score 1.
country['rank_score'] = 51 - country['daily_rank']
country['chart_scope_type'] = np.where(
    country['country'] == 'GLOBAL', 'global', 'country'
)
country['album_release_year'] = country['album_release_date'].dt.year

In [9]:
historical[['track_name', 'year', 'mood_label', 'duration_min', 'release_decade']].head()

,track_name,year,mood_label,duration_min,release_decade
0,Oops!...I Did It Again,2000,Euphoric,3.519333,2000
1,All The Small Things,2000,Euphoric,2.784450,2000
2,Breathe,2000,Melancholic,4.175783,2000
3,It's My Life,2000,Euphoric,3.741550,2000
4,Bye Bye Bye,2000,Euphoric,3.340000,2000


## 4. Mark single-genre and multi-genre tracks

A track can appear with more than one genre label. We keep all of them for analysis, but later the main single-label genre classifier will use tracks that have exactly one genre.

In [10]:
genre_count = genre.groupby('track_id')['track_genre'].nunique()

genre['genre_count'] = genre['track_id'].map(genre_count)
genre['is_single_genre_track'] = genre['genre_count'] == 1

print("Unique tracks:", genre['track_id'].nunique())
print("Single-genre tracks:", genre.loc[genre['is_single_genre_track'], 'track_id'].nunique())
print("Multi-genre tracks:", genre.loc[~genre['is_single_genre_track'], 'track_id'].nunique())

Unique tracks: 89741
Single-genre tracks: 73442
Multi-genre tracks: 16299


In [11]:
genre[['track_id', 'track_name', 'track_genre', 'genre_count', 'is_single_genre_track']].head(10)

,track_id,track_name,track_genre,genre_count,is_single_genre_track
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,acoustic,4,False
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,acoustic,2,False
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,acoustic,1,True
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,acoustic,1,True
4,5vjLSffimiIP26QG5WcN2K,Hold On,acoustic,1,True
5,01MVOl9KtVTNfFiBU9I7dc,Days I Will Remember,acoustic,2,False
6,6Vc5wAMmXdKIAM7WUoEb7N,Say Something,acoustic,2,False
7,1EzrEOXmMH3G43AXT1y7pA,I'm Yours,acoustic,2,False
8,0IktbUcnAGrvD03AWnz3Q8,Lucky,acoustic,1,True
9,7k9GuJYLp2AzqokyEdwEw2,Hunger,acoustic,2,False


## 5. Build a simple track catalog

The catalog keeps one display row per track. Genre information is merged separately so repeated chart/year observations do not create duplicate catalog rows.

In [12]:
# THIS CATALOUGE IS FOR MOOD AND OTHER STUFFS, NOT POPULARITU BY COUNTRY
# Start with one row per track from each source.
genre_catalog = genre.drop_duplicates('track_id').copy()
historical_catalog = historical.drop_duplicates('track_id').copy()
country_catalog = country.drop_duplicates('track_id').copy()

# The historical source does not contain an explicit-content field.
historical_catalog['explicit'] = pd.NA

catalog_columns = [
    'track_id', 'track_name', 'artist_name', 'album_name', 'explicit',
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_ms', 'time_signature'
]

# Genre data is first because it contains the largest track collection.
track_catalog = pd.concat([
    genre_catalog[catalog_columns],
    historical_catalog[catalog_columns],
    country_catalog[catalog_columns]
], ignore_index=True)

track_catalog = track_catalog.drop_duplicates('track_id').reset_index(drop=True)

In [13]:
# Collect the genres belonging to each track.
genre_lists = (
    genre.groupby('track_id')['track_genre']
    .apply(lambda values: sorted(set(values)))
    .reset_index(name='genre_list')
)

genre_lists['genre_count'] = genre_lists['genre_list'].apply(len)
genre_lists['genres'] = genre_lists['genre_list'].apply(lambda x: ', '.join(x))
genre_lists['primary_genre'] = genre_lists['genre_list'].apply(lambda x: x[0] if len(x) > 0 else np.nan)

genre_lists = genre_lists.drop(columns='genre_list')

track_catalog = track_catalog.merge(
    genre_lists,
    on='track_id',
    how='left'
)

In [14]:
# These flags show which source datasets contain each track.
genre_ids = set(genre['track_id'])
historical_ids = set(historical['track_id'])
country_ids = set(country['track_id'])

track_catalog['in_genre_dataset'] = track_catalog['track_id'].isin(genre_ids)
track_catalog['in_historical_dataset'] = track_catalog['track_id'].isin(historical_ids)
track_catalog['in_country_dataset'] = track_catalog['track_id'].isin(country_ids)

track_catalog = track_catalog.sort_values('track_id').reset_index(drop=True)

print("Tracks in catalog:", len(track_catalog))
track_catalog.head()

Tracks in catalog: 115657


,track_id,track_name,artist_name,album_name,explicit,danceability,energy,key,loudness,mode,...,valence,tempo,duration_ms,time_signature,genre_count,genres,primary_genre,in_genre_dataset,in_historical_dataset,in_country_dataset
0,0000vdREvCVMxbQTkS888c,Lolly,Rill,Lolly,True,0.910,0.374,8,-9.844,0,...,0.432,104.042,160725,4,1.0,german,german,True,False,False
1,000CC8EParg64OmTxVnZ0p,It's All Coming Back To Me Now (Glee Cast Vers...,Glee Cast,Glee Love Songs,False,0.269,0.516,0,-7.361,1,...,0.341,178.174,322933,4,1.0,club,club,True,False,False
2,000Iz0K615UepwSJ5z2RE5,Böxig Leise - Pig & Dan Remix,Paul Kalkbrenner;Pig&Dan,X,False,0.686,0.560,5,-13.264,0,...,0.108,119.997,515360,4,1.0,minimal-techno,minimal-techno,True,False,False
3,000RDCYioLteXcutOjeweY,Teeje Week,Jordan Sandhu,Teeje Week,False,0.679,0.770,0,-3.537,1,...,0.839,161.721,190203,4,1.0,hip-hop,hip-hop,True,False,False
4,000n6Lx4yqUAslF1x3JeFY,béke,"Azahriah, DESH, Young Fly, Lord Panamo, Copy Con",tripq,False,0.676,0.728,9,-7.661,0,...,0.938,174.089,315862,4,NaN,NaN,NaN,False,False,True


## 6. Save the mood and feature definitions

In [15]:
mood_definition = {
    'valence_threshold': 0.50,
    'energy_threshold': 0.50,
    'moods': {
        'Euphoric': 'high valence + high energy',
        'Peaceful': 'high valence + low energy',
        'Aggressive': 'low valence + high energy',
        'Melancholic': 'low valence + low energy'
    },
    'colors': {
        'Euphoric': '#F4C95D',
        'Peaceful': '#67C6C3',
        'Aggressive': '#E85D5D',
        'Melancholic': '#6C70B5'
    }
}

feature_schema = {
    'audio_features': [
        'danceability', 'energy', 'key', 'loudness', 'mode',
        'speechiness', 'acousticness', 'instrumentalness',
        'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature'
    ],
    'mood_model_features': [
        'danceability', 'key', 'loudness', 'mode', 'speechiness',
        'acousticness', 'instrumentalness', 'liveness', 'tempo',
        'duration_ms', 'time_signature'
    ],
    'note': 'Valence and energy are excluded from the later mood classifier because they define mood_label.'
}

os.makedirs('config', exist_ok=True)

with open('config/mood_definition.json', 'w') as f:
    json.dump(mood_definition, f, indent=4)

with open('config/feature_schema.json', 'w') as f:
    json.dump(feature_schema, f, indent=4)

## 7. Save the Phase 2 datasets

In [16]:
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports/feature_engineering', exist_ok=True)

# Save easy-to-inspect compressed CSV copies.
historical.to_csv('data/processed/historical_tracks.csv.gz', index=False, compression='gzip')
genre.to_csv('data/processed/genre_tracks.csv.gz', index=False, compression='gzip')
country.to_csv('data/processed/country_chart_observations.csv.gz', index=False, compression='gzip')

# naya track catalog also saved
track_catalog.to_csv('data/processed/track_catalog.csv.gz', index=False, compression='gzip')

# Save Parquet files when pyarrow is installed.
try:
    historical.to_parquet('data/processed/historical_tracks.parquet', index=False)
    genre.to_parquet('data/processed/genre_tracks.parquet', index=False)
    country.to_parquet('data/processed/country_chart_observations.parquet', index=False)
    track_catalog.to_parquet('data/processed/track_catalog.parquet', index=False)
    print("Parquet files saved.")
except ImportError:
    print("PyArrow is not installed, so only the CSV copies were saved.")

phase2_summary = {
    'historical_rows': int(len(historical)),
    'genre_rows': int(len(genre)),
    'country_rows': int(len(country)),
    'track_catalog_rows': int(len(track_catalog)),
    'unique_genre_tracks': int(genre['track_id'].nunique()),
    'single_genre_tracks': int(genre.loc[genre['is_single_genre_track'], 'track_id'].nunique()),
    'multi_genre_tracks': int(genre.loc[~genre['is_single_genre_track'], 'track_id'].nunique())
}

with open('reports/feature_engineering/feature_engineering_summary.json', 'w') as f:
    json.dump(phase2_summary, f, indent=4)

print("Phase 2 complete.")

Parquet files saved.
Phase 2 complete.
